# **Merchant**

## Security warning before sharing\n
\n
This notebook contains hardcoded Telegram API details, a phone number, Google Sheets IDs, channel links, local paths, and references to a Google service-account key. Do not include `khemra_account.json` or the `tg_sessions/` folder in any ZIP sent to another person. Clear notebook outputs and replace credentials with placeholders or environment variables before sharing.

## Retail_Banking 


In [ ]:
import nest_asyncio
import asyncio
import random
import time
import os
import sys
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd
import pytz
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from telethon import TelegramClient
from telethon.tl.functions.messages import GetHistoryRequest
from telethon.errors import FloodWaitError, RPCError
from telethon.tl.types import PeerUser, PeerChannel, PeerChat
import re

# Apply nest_asyncio to fix the event loop issue
nest_asyncio.apply()

api_id = 22365302
api_hash = 'df22eea81948788953b28b8112ab926a'
#session_name = 'telegram_session'
SESSION_DIR = "./tg_sessions"
os.makedirs(SESSION_DIR, exist_ok=True)
session_name = os.path.join(SESSION_DIR, "geo_scraper")
phone_number = '+855885478958'

# Telegram channel/group links to scrape for Retail_Banking messages.
TARGET_CHANNELS = [
    "https://t.me/+m-6mzB80O_ZkZmFl" # Add or remove links here when the Retail_Banking source channels change.
]

# ==============================================================================
# --- GOOGLE SHEETS CONFIG ---
# This section controls where the scraped Retail_Banking records are uploaded.
# IMPORTANT: Do not share the real service-account JSON file with other people.
SHEET_JSON = r"C:\Users\sombath.kim\Documents\Retail_Banking\khemra_account copy.json" # PLease input the path to your Google service-account JSON file here
SHEET_ID = "1wM7DTHizhg_A3h0qV3EhX4os4hk46uolW-ESQSJkgZs" # Please input the Google Spreadsheet ID here
WORKSHEET_NAME = "Retail_Banking" # WORKSHEET_NAME is the tab name inside that spreadsheet where data will be written.

TESTING_MODE = True

# ============================================================
# OLD STYLE: MANUAL DATE BETWEEN
# This section manually defines the Telegram message date window to scrape.
# MIN_DATE is the start time; NOW is the end time, using Cambodia timezone.
# Change only these 2 lines when you want another period.
# Example: scrape from 8 PM yesterday to 7 PM today by updating MIN_DATE and NOW.
# ============================================================
CAMBODIA_TZ = pytz.timezone("Asia/Phnom_Penh")
MIN_DATE = datetime(2026, 6, 24, 20, 0, 0) # Only scrape messages newer than this date
NOW = datetime(2026, 6, 25, 19, 0, 0)

# Large numeric limits let the existing scraper logic scan all branches in the date window.
if TESTING_MODE:
    HISTORY_LIMIT = 100
    MAX_REQUESTS_PER_RUN = 1000000
    MAX_MESSAGES = 1000000
    DELAY_MIN = 4.0
    DELAY_MAX = 8.0
    ENTITY_DELAY_MIN = 1.5
    ENTITY_DELAY_MAX = 2.5
else:
    HISTORY_LIMIT = 100
    MAX_REQUESTS_PER_RUN = 1000000
    MAX_MESSAGES = 1000000
    DELAY_MIN = 2.0
    DELAY_MAX = 4.0
    ENTITY_DELAY_MIN = 0.5
    ENTITY_DELAY_MAX = 1.0

# Match text message and separate location message from same sender
LOCATION_MATCH_WINDOW_MINUTES = 15

# Global in-memory cache for sender information
sender_cache = {}

# ==============================================================================
# === TEST SAFE SCRAPER ===
# ==============================================================================
class TestSafeScraper:
    """Manages global limits for testing mode."""
    def __init__(self):
        self.request_count = 0
        self.start_time = time.time()
        self.message_count = 0

    def can_continue(self):
        if self.request_count >= MAX_REQUESTS_PER_RUN:
            return False
        if self.message_count >= MAX_MESSAGES:
            return False
        return True

scraper = TestSafeScraper()

# ==============================================================================
# === FIELD DEFINITIONS + LABEL-BASED PARSER ===
# ==============================================================================
FIELD_ORDER = [
    "Type", "Call Plan", "Direction", "Client Name", "Contact", "Category"
]

SHEET_COLUMNS = [
    "Source_Channel", "Sender_ID", "Sender_Name", "Type", "Call_Plan", "Direction",
    "Client_Name", "Contact", "Category", "Message_Date", "Latitude", "Longitude",
    "Raw_Text", "Has_Image", "Has_Location"
]

FIELD_LABEL_ALIASES = {
    "type": "Type",
    "call plan": "Call Plan",
    "direction": "Direction",
    "client name": "Client Name",
    "client": "Client Name",
    "contact": "Contact",
    "category": "Category",
}

LABEL_PATTERN = (
    r"Client\s*Name|Call\s*Plan|Direction|Category|Contact|Type"
)
LABEL_SEPARATOR_PATTERN = r"\s*[:：\-–—]\s*"
LABEL_LINE_RE = re.compile(
    rf"^\s*(?:[-*•]\s*)?(?P<label>{LABEL_PATTERN})\s*{LABEL_SEPARATOR_PATTERN}\s*(?P<value>.*)$",
    re.IGNORECASE,
)
INLINE_LABEL_RE = re.compile(
    rf"(?<!^)(?<!\n)\s+(?=(?:{LABEL_PATTERN})\s*{LABEL_SEPARATOR_PATTERN})",
    re.IGNORECASE,
)
ONLY_LABEL_RE = re.compile(
    rf"^(?:{LABEL_PATTERN})\s*{LABEL_SEPARATOR_PATTERN}?\s*$",
    re.IGNORECASE,
)


def normalize_field_label(label):
    cleaned = re.sub(r"\s+", " ", str(label or "").strip().lower())
    return FIELD_LABEL_ALIASES.get(cleaned)


def prepare_structured_text(text):
    normalized = str(text or "").replace("\r\n", "\n").replace("\r", "\n")
    normalized = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", normalized)
    # Preserve explicit line breaks and split inline labels to avoid shifted values.
    return INLINE_LABEL_RE.sub("\n", normalized)


def clean_extracted_value(value):
    cleaned = re.sub(r"\s+", " ", str(value or "").strip())

    if not cleaned:
        return ""

    if ONLY_LABEL_RE.match(cleaned):
        return ""

    lower_cleaned = cleaned.lower()

    for alias in FIELD_LABEL_ALIASES.keys():
        if lower_cleaned == alias:
            return ""

    return cleaned


def extract_info_from_text(text):
    data = {field: "" for field in FIELD_ORDER}
    normalized_text = prepare_structured_text(text)

    current_key = None
    current_value_lines = []

    for raw_line in normalized_text.splitlines():
        line = raw_line.rstrip()
        if not line.strip():
            if current_key is not None:
                current_value_lines.append("")
            continue

        match = LABEL_LINE_RE.match(line)
        if match:
            if current_key is not None:
                data[current_key] = clean_extracted_value("\n".join(current_value_lines))

            current_key = normalize_field_label(match.group("label"))
            current_value_lines = [match.group("value") or ""] if current_key else []
            continue

        if current_key is not None:
            if not ONLY_LABEL_RE.match(line.strip()):
                current_value_lines.append(line.strip())
            continue

    if current_key is not None:
        data[current_key] = clean_extracted_value("\n".join(current_value_lines))

    return data


def has_structured_fields(text):
    normalized_text = prepare_structured_text(text)
    return any(
        LABEL_LINE_RE.match(raw_line.strip())
        for raw_line in normalized_text.splitlines()
    )


def get_sender_id(from_id):
    """Safely extracts a string ID from the telethon Peer object."""
    if isinstance(from_id, PeerUser):
        return str(from_id.user_id)
    elif isinstance(from_id, PeerChannel):
        return str(from_id.channel_id)
    elif isinstance(from_id, PeerChat):
        return str(from_id.chat_id)
    return str(from_id)


async def get_sender_info(client, from_id):
    """Fetch sender name using cache."""
    sender_id = get_sender_id(from_id)

    if sender_id in sender_cache:
        return sender_id, sender_cache[sender_id]

    try:
        sender = await client.get_entity(from_id)
        scraper.request_count += 1

        first = getattr(sender, "first_name", "") or ""
        last = getattr(sender, "last_name", "") or ""
        sender_name = (first + " " + last).strip() or getattr(
            sender, "username", f"ID:{sender_id}"
        )

        sender_cache[sender_id] = sender_name

        delay = random.uniform(ENTITY_DELAY_MIN, ENTITY_DELAY_MAX)
        await asyncio.sleep(delay)

        return sender_id, sender_name
    except Exception:
        sender_name = f"ID:{sender_id}"
        sender_cache[sender_id] = sender_name
        return sender_id, sender_name


async def testing_delay(delay_min=DELAY_MIN, delay_max=DELAY_MAX):
    delay = random.uniform(delay_min, delay_max)
    await asyncio.sleep(delay)


def extract_geo_from_message(msg):
    """
    Telegram location is usually inside msg.media.geo, not msg.geo
    """
    media = getattr(msg, "media", None)
    if not media:
        return None, None

    geo = getattr(media, "geo", None)
    if geo and hasattr(geo, "lat") and hasattr(geo, "long"):
        return geo.lat, geo.long

    return None, None


def sanitize_dataframe_for_sheets(df):
    """
    Remove NaN / NaT / inf before sending to Google Sheets
    """
    df = df.copy()
    df = df.replace([float("inf"), float("-inf")], "")
    df = df.astype(object)
    df = df.where(pd.notnull(df), "")
    return df


def column_alias_key(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name or "").strip().lower()).strip("_")

COLUMN_ALIASES = {column_alias_key(column): column for column in SHEET_COLUMNS}
COLUMN_ALIASES.update({
    "call_plan": "Call_Plan",
    "client": "Client_Name",
    "client_name": "Client_Name",
})



def normalize_sheet_column(column):
    return COLUMN_ALIASES.get(column_alias_key(column), "")


def align_to_sheet_columns(df):
    aligned = pd.DataFrame(index=df.index)

    for col_index, original_column in enumerate(df.columns):
        target_column = normalize_sheet_column(original_column)
        if not target_column:
            continue

        values = df.iloc[:, col_index]
        if target_column in aligned.columns:
            current_has_value = aligned[target_column].astype(str).str.strip() != ""
            aligned[target_column] = aligned[target_column].where(current_has_value, values)
        else:
            aligned[target_column] = values

    return aligned.reindex(columns=SHEET_COLUMNS, fill_value="")


def pop_best_candidate(candidates, target_dt, max_gap_minutes):
    """
    Find nearest message in time within allowed window
    """
    if not candidates:
        return None

    best_idx = None
    best_gap = None

    for idx, item in enumerate(candidates):
        gap = abs((item["date"] - target_dt).total_seconds())
        if gap <= max_gap_minutes * 60:
            if best_gap is None or gap < best_gap:
                best_gap = gap
                best_idx = idx

    if best_idx is None:
        return None

    return candidates.pop(best_idx)

# ==============================================================================
# === CORE SCRAPING AND PROCESSING LOGIC ===
# ==============================================================================
async def scrape_single_target(client, target_url):
    """
    Scrape structured customer text and attach lat/long if:
    1. location is in same message
    2. location is in a nearby separate message from same sender
    """
    messages_data = []
    offset_id = 0
    all_messages = []
    batch_count = 0

    pending_text_records = defaultdict(list)
    pending_location_messages = defaultdict(list)

    skipped_too_old = 0
    skipped_too_new = 0
    skipped_no_text = 0
    skipped_no_structured = 0

    try:
        entity = await client.get_entity(target_url)
        scraper.request_count += 1
        source_channel_name = getattr(entity, "title", target_url)
        print(f"\n🎯 Starting scrape for: {source_channel_name}")
        await testing_delay()

        min_date_tz = CAMBODIA_TZ.localize(MIN_DATE)
        now_tz = CAMBODIA_TZ.localize(NOW)

        print(f"📅 Filter from {min_date_tz.strftime('%Y-%m-%d %H:%M:%S')} to {now_tz.strftime('%Y-%m-%d %H:%M:%S')}")

        while scraper.can_continue():
            try:
                history = await client(
                    GetHistoryRequest(
                        peer=entity,
                        limit=HISTORY_LIMIT,
                        offset_id=offset_id,
                        offset_date=None,
                        max_id=0,
                        min_id=0,
                        add_offset=0,
                        hash=0,
                    )
                )
                scraper.request_count += 1

            except FloodWaitError as e:
                wait_time = e.seconds + random.randint(5, 15)
                print(f"🛑 FloodWait: Wait {wait_time}s.")
                await asyncio.sleep(wait_time)
                continue
            except RPCError as e:
                print(f"⚠️ RPC Error: {e}")
                await asyncio.sleep(20)
                continue

            messages = history.messages
            if not messages:
                print("✅ No more messages.")
                break

            all_messages.extend(messages)
            scraper.message_count += len(messages)
            offset_id = messages[-1].id
            batch_count += 1

            for msg in messages:
                if not scraper.can_continue():
                    break

                if not getattr(msg, "date", None):
                    continue

                msg_date_tz = msg.date.astimezone(CAMBODIA_TZ)

                if msg_date_tz < min_date_tz:
                    skipped_too_old += 1
                    continue
                if msg_date_tz > now_tz:
                    skipped_too_new += 1
                    continue

                sender_id, sender_name = None, None
                if getattr(msg, "from_id", None):
                    sender_id, sender_name = await get_sender_info(client, msg.from_id)

                raw_text = msg.message or getattr(msg, "caption", "") or ""
                text = raw_text.strip()
                lat, lon = extract_geo_from_message(msg)
                has_location = lat is not None and lon is not None
                has_media = bool(getattr(msg, "media", None))

                # Case 1: location-only message
                if has_location and not text:
                    if sender_id:
                        matched_text = pop_best_candidate(
                            pending_text_records.get(sender_id, []),
                            msg_date_tz,
                            LOCATION_MATCH_WINDOW_MINUTES
                        )
                        if matched_text is not None:
                            idx = matched_text["record_index"]
                            messages_data[idx]["Latitude"] = lat
                            messages_data[idx]["Longitude"] = lon
                            messages_data[idx]["Has_Location"] = True
                        else:
                            pending_location_messages[sender_id].append({
                                "date": msg_date_tz,
                                "lat": lat,
                                "lon": lon,
                            })
                    continue

                if not text:
                    skipped_no_text += 1
                    continue

                if not has_structured_fields(text):
                    skipped_no_structured += 1
                    continue

                extracted = extract_info_from_text(text)

                customer_data = {
                    "Source_Channel": source_channel_name,
                    "Sender_ID": sender_id or "",
                    "Sender_Name": sender_name or "",
                    "Type": extracted.get("Type", ""),
                    "Call_Plan": extracted.get("Call Plan", ""),
                    "Direction": extracted.get("Direction", ""),
                    "Client_Name": extracted.get("Client Name", ""),
                    "Contact": extracted.get("Contact", ""),
                    "Category": extracted.get("Category", ""),
                    "Message_Date": msg_date_tz.strftime("%Y-%m-%d %H:%M:%S"),
                    "Latitude": "",
                    "Longitude": "",
                    "Raw_Text": raw_text,
                    "Has_Image": bool(has_media and not has_location),
                    "Has_Location": False,
                }

                # Same message contains location
                if has_location:
                    customer_data["Latitude"] = lat
                    customer_data["Longitude"] = lon
                    customer_data["Has_Location"] = True

                # Otherwise match with separate nearby location message
                elif sender_id:
                    matched_location = pop_best_candidate(
                        pending_location_messages.get(sender_id, []),
                        msg_date_tz,
                        LOCATION_MATCH_WINDOW_MINUTES
                    )
                    if matched_location is not None:
                        customer_data["Latitude"] = matched_location["lat"]
                        customer_data["Longitude"] = matched_location["lon"]
                        customer_data["Has_Location"] = True

                messages_data.append(customer_data)

                # If still no location, keep text row waiting
                if sender_id and not customer_data["Has_Location"]:
                    pending_text_records[sender_id].append({
                        "date": msg_date_tz,
                        "record_index": len(messages_data) - 1,
                    })

            print(f"📦 Batch {batch_count}: {len(messages_data)} records so far")

            # Stop when oldest fetched message is older than min_date
            if messages[-1].date.astimezone(CAMBODIA_TZ) < min_date_tz:
                break

            await testing_delay()

        print(f"✅ Done: {len(all_messages)} messages → {len(messages_data)} customer records")
        print(f"   • Skipped too old: {skipped_too_old}")
        print(f"   • Skipped too new: {skipped_too_new}")
        print(f"   • Skipped no text: {skipped_no_text}")
        print(f"   • Skipped no structured fields: {skipped_no_structured}")
        return messages_data

    except Exception as e:
        print(f"❌ Scraping failed for {target_url}: {e}")
        return messages_data

# ==============================================================================
# === GOOGLE SHEETS FUNCTIONALITY ===
# ==============================================================================

def clean_and_convert_dataframe(df):
    """Optional cleaner if you want to inspect data locally."""
    df_clean = df.copy()

    numeric_columns = ['Latitude', 'Longitude']
    for col in numeric_columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str)
            df_clean[col] = df_clean[col].str.replace('%', '', regex=False)
            df_clean[col] = df_clean[col].str.replace(',', '', regex=False)
            df_clean[col] = df_clean[col].str.replace('$', '', regex=False)
            df_clean[col] = df_clean[col].str.replace(' ', '', regex=False)
            df_clean[col] = df_clean[col].replace(['', 'nan', 'None', 'N/A', 'n/a'], None)
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    if 'Message_Date' in df_clean.columns:
        df_clean['Message_Date'] = pd.to_datetime(df_clean['Message_Date'], errors='coerce')

    text_columns = [
        'Source_Channel', 'Sender_ID', 'Sender_Name', 'Type', 'Call_Plan',
        'Direction', 'Client_Name', 'Contact', 'Category', 'Raw_Text'
    ]
    for col in text_columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('').astype(str)

    return df_clean


def log_validation_warnings(df):
    """Log parser warnings before uploading to Google Sheets."""
    required_columns = ["Type", "Call_Plan", "Direction", "Client_Name", "Contact", "Category"]
    for row_index, row in df.iterrows():
        missing = [column for column in required_columns if not str(row.get(column, "")).strip()]
        if missing:
            print(f"Validation warning row {row_index + 1}: Missing {', '.join(missing)}")



def push_to_google_sheet(data):
    """Push processed DataFrame to Google Sheets."""
    if not data:
        print("⚠️ No data to push to Google Sheets.")
        return

    df = pd.DataFrame(data)
    df = align_to_sheet_columns(df)
    df = sanitize_dataframe_for_sheets(df)
    log_validation_warnings(df)

    try:
        print("\n⚙️ Starting Google Sheets authentication...")
        scope = [
            "https://spreadsheets.google.com/feeds",
            "https://www.googleapis.com/auth/drive"
        ]

        creds = ServiceAccountCredentials.from_json_keyfile_name(SHEET_JSON, scope)
        client = gspread.authorize(creds)

        sheet = client.open_by_key(SHEET_ID).worksheet(WORKSHEET_NAME)

        existing_values = sheet.get_all_values()
        if len(existing_values) > 1:
            existing_header = existing_values[0]
            existing_rows = [
                (row + [""] * len(existing_header))[:len(existing_header)]
                for row in existing_values[1:]
                if any(str(cell).strip() for cell in row)
            ]
            existing_df = pd.DataFrame(existing_rows, columns=existing_header)
            existing_df = align_to_sheet_columns(existing_df)
            print(f"📄 Existing rows in sheet: {len(existing_df)}")

            combined_df = pd.concat([existing_df, df], ignore_index=True)
            combined_df.drop_duplicates(
                subset=["Sender_Name", "Message_Date"],
                keep="last",
                inplace=True
            )
        else:
            print("🆕 Sheet is empty. Creating new one.")
            combined_df = df.copy()

        combined_df = align_to_sheet_columns(combined_df)
        combined_df = sanitize_dataframe_for_sheets(combined_df)

        sheet.clear()
        sheet.update([SHEET_COLUMNS] + combined_df[SHEET_COLUMNS].values.tolist(), value_input_option="RAW")

        print(f"✅ Pushed {len(df)} rows to Google Sheets (Worksheet: {WORKSHEET_NAME}) at {NOW.strftime('%Y-%m-%d %H:%M:%S')}")
    except Exception as e:
        print(f"❌ Failed to push data to Google Sheets: {e}")
        print("   Check your SHEET_JSON path, SHEET_ID, and ensure the service account has edit access to the sheet.")

# ==============================================================================
# === MAIN EXECUTION ===
# ==============================================================================
async def main():
    """Main function to orchestrate scraping and data push."""
    global scraper
    all_data = []

    print("🧪 ===== TELEGRAM SCRAPER - MULTI-TARGET EXECUTION =====")
    print(f"📋 Global Limits: {MAX_REQUESTS_PER_RUN} requests, {MAX_MESSAGES} messages (TESTING={TESTING_MODE})")
    print(f"📅 Scrape window: {MIN_DATE.strftime('%Y-%m-%d %H:%M:%S')}  ->  {NOW.strftime('%Y-%m-%d %H:%M:%S')}")

    try:
        if os.path.exists(f"{session_name}.session"):
            file_age = time.time() - os.path.getmtime(f"{session_name}.session")
            if file_age < 300 and TESTING_MODE:
                print(f"🕒 Safety: Last run was {file_age:.0f}s ago. Waiting 10s...")
                await asyncio.sleep(10)

        async with TelegramClient(session_name, api_id, api_hash) as client:
            await client.start(phone=phone_number)
            scraper.request_count += 1

            if not await client.is_user_authorized():
                print("❌ Not authorized. Please check verification code in your console.")
                return

            print("🔗 Connected to Telegram. Starting channel loop.")

            for target_url in TARGET_CHANNELS:
                if not scraper.can_continue():
                    print("🛑 Global API/Message limit reached. Stopping multi-target run.")
                    break

                channel_data = await scrape_single_target(client, target_url)
                all_data.extend(channel_data)

            print("\n=======================================================")
            print("🚀 PUSHING DATA TO GOOGLE SHEETS")
            print("=======================================================")
            push_to_google_sheet(all_data)

            print(f"\n🎉 EXECUTION COMPLETED!")
            print(f"📊 Final Statistics:")
            print(f"  • Total API Requests: {scraper.request_count}/{MAX_REQUESTS_PER_RUN}")
            print(f"  • Total Messages Fetched: {scraper.message_count}")
            print(f"  • Total Structured Records Found: {len(all_data)}")
            print(f"  • Time Elapsed: {time.time() - scraper.start_time:.1f}s")

    except Exception as e:
        print(f"❌ Global Execution failed: {e}")
        print("💡 Ensure your API keys, phone number, and channel links are correct.")

# Run
if __name__ == "__main__":
    if 'last_run' in globals():
        time_since_last = time.time() - globals()['last_run']
        if time_since_last < 60:
            print(f"🚨 Safety: Wait {60-time_since_last:.0f}s before re-running")
            exit()

    globals()['last_run'] = time.time()
    asyncio.run(main())

🧪 ===== TELEGRAM SCRAPER - MULTI-TARGET EXECUTION =====
📋 Global Limits: 1000000 requests, 1000000 messages (TESTING=True)
📅 Scrape window: 2026-06-24 20:00:00  ->  2026-06-25 19:00:00
